# W2-D1: Alert Correlation Pipeline

Author: ThanhTam

Goal: build a 3-layer correlator: fingerprint, time-window session, and topology-aware grouping.


## Cell 1 - Imports and Load Dataset


In [1]:
import json
import os
from datetime import datetime
from collections import defaultdict, deque

ALERT_FILE = "dataset/alerts_sample.jsonl"
GRAPH_FILE = "dataset/services.json"

with open(ALERT_FILE, "r", encoding="utf-8") as f:
    alerts = [json.loads(line) for line in f if line.strip()]

with open(GRAPH_FILE, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

service_names = {svc["name"] for svc in graph_data.get("services", [])}
service_graph = defaultdict(set)
for name in service_names:
    service_graph[name]

# Only service-to-service edges are used for incident grouping. Backing stores are
# context, but using them as merge nodes can over-correlate independent alerts.
for edge in graph_data.get("edges", []):
    src, dst = edge["from"], edge["to"]
    if src in service_names and dst in service_names:
        service_graph[src].add(dst)
        service_graph[dst].add(src)

print(f"Loaded {len(alerts)} alerts")
print(f"Service graph: {len(service_graph)} service nodes, {sum(len(v) for v in service_graph.values()) // 2} service edges")
print(json.dumps(alerts[0], indent=2))

Loaded 20 alerts
Service graph: 10 service nodes, 10 service edges
{
  "id": "a-0001",
  "ts": "2026-06-12T09:42:01Z",
  "service": "payment-svc",
  "metric": "db_connection_pool_used_ratio",
  "severity": "warn",
  "value": 0.85,
  "threshold": 0.8,
  "labels": {
    "env": "prod",
    "region": "ap-southeast-1"
  }
}


## Cell 2 - Define Correlation Functions


In [2]:
SEVERITY_RANK = {"info": 0, "warn": 1, "crit": 2}

def parse_ts(ts: str) -> datetime:
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))

def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

def max_severity(alerts_in_group: list[dict]) -> str:
    return max(alerts_in_group, key=lambda a: SEVERITY_RANK.get(a["severity"], 0))["severity"]

def hop_distance(graph: dict[str, set[str]], source: str, target: str, max_hop: int) -> int | None:
    if source == target:
        return 0
    queue = deque([(source, 0)])
    seen = {source}
    while queue:
        node, depth = queue.popleft()
        if depth >= max_hop:
            continue
        for nxt in graph.get(node, set()):
            if nxt == target:
                return depth + 1
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, depth + 1))
    return None

def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: parse_ts(a["ts"]))
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        gap = (parse_ts(alert["ts"]) - parse_ts(groups[-1][-1]["ts"])).total_seconds()
        if gap <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups

def topology_group(alerts: list[dict], graph: dict[str, set[str]], max_hop: int = 2) -> list[list[dict]]:
    """Group around the first alert service of each incident candidate.

    This avoids transitive over-merge: A can be close to B and B close to C,
    but A and C should not automatically be in the same incident when they are
    farther than max_hop from the incident center.
    """
    clusters = []
    for alert in sorted(alerts, key=lambda a: parse_ts(a["ts"])):
        service = alert["service"]
        matched = False
        for cluster in clusters:
            center = cluster["center"]
            distance = hop_distance(graph, center, service, max_hop)
            if distance is not None and distance <= max_hop:
                cluster["alerts"].append(alert)
                matched = True
                break
        if not matched:
            clusters.append({"center": service, "alerts": [alert]})
    return [cluster["alerts"] for cluster in clusters]

def correlate(alerts: list[dict], graph: dict[str, set[str]], gap_sec: int = 120, max_hop: int = 2) -> list[dict]:
    clusters = []
    for s_idx, session_alerts in enumerate(session_groups(alerts, gap_sec=gap_sec)):
        for g_idx, group in enumerate(topology_group(session_alerts, graph, max_hop=max_hop)):
            clusters.append({
                "cluster_id": f"c-{s_idx:03d}-{g_idx:03d}",
                "alert_count": len(group),
                "services": sorted({a["service"] for a in group}),
                "time_range": [min(a["ts"] for a in group), max(a["ts"] for a in group)],
                "max_severity": max_severity(group),
                "fingerprints": list(dict.fromkeys(fingerprint(a) for a in group)),
            })
    return clusters

print("Correlation functions are ready")
print("gap_sec default = 120")
print("max_hop default = 2")

Correlation functions are ready
gap_sec default = 120
max_hop default = 2


## Cell 3 - Run Pipeline


In [3]:
GAP_SEC = 120
MAX_HOP = 2

output_clusters = correlate(alerts, service_graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)

print(f"Input alerts: {len(alerts)}")
print(f"Output clusters: {len(output_clusters)}")
for cluster in output_clusters:
    print(f"{cluster['cluster_id']} | alerts={cluster['alert_count']} | max_severity={cluster['max_severity']} | services={cluster['services']}")
    print(f"  time_range={cluster['time_range']}")
    print(f"  fingerprints={cluster['fingerprints']}")

Input alerts: 20
Output clusters: 3
c-000-000 | alerts=18 | max_severity=crit | services=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
  time_range=['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z']
  fingerprints=['payment-svc|db_connection_pool_used_ratio|warn', 'payment-svc|db_connection_pool_used_ratio|crit', 'payment-svc|latency_p99_ms|crit', 'payment-svc|error_rate|warn', 'checkout-svc|latency_p99_ms|warn', 'checkout-svc|downstream_payment_error_rate|crit', 'edge-lb|upstream_5xx_rate|warn', 'cart-svc|latency_p99_ms|warn', 'notification-svc|queue_lag_ms|warn', 'checkout-svc|request_drop_rate|crit', 'edge-lb|p99_latency_ms|crit', 'checkout-svc|latency_p99_ms|crit', 'payment-svc|error_rate|crit', 'notification-svc|queue_depth|crit', 'edge-lb|upstream_5xx_rate|crit']
c-000-001 | alerts=1 | max_severity=warn | services=['recommender-svc']
  time_range=['2026-06-12T09:45:10Z', '2026-06-12T09:45:10Z']
  fingerprints=['recommender-svc|cpu_utilization|warn']
c-0

## Cell 4 - Write Required JSON Output


In [4]:
summary = {
    "input_alerts": len(alerts),
    "output_clusters": len(output_clusters),
    "reduction_ratio": round(1 - len(output_clusters) / len(alerts), 2) if alerts else 0.0,
    "clusters": output_clusters,
}

os.makedirs("results", exist_ok=True)
with open("results/cluster_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("Written results/cluster_summary.json")
print(json.dumps({k: summary[k] for k in ["input_alerts", "output_clusters", "reduction_ratio"]}, indent=2))

Written results/cluster_summary.json
{
  "input_alerts": 20,
  "output_clusters": 3,
  "reduction_ratio": 0.85
}


## Cell 5 - Validate Acceptance Criteria


In [5]:
errors = []

if not os.path.exists("results/cluster_summary.json"):
    errors.append("results/cluster_summary.json is missing")
else:
    with open("results/cluster_summary.json", "r", encoding="utf-8") as f:
        loaded = json.load(f)
    for field in ["input_alerts", "output_clusters", "reduction_ratio", "clusters"]:
        if field not in loaded:
            errors.append(f"missing top-level field: {field}")
    if loaded.get("reduction_ratio", 0) < 0.5:
        errors.append("reduction_ratio must be >= 0.5")
    for cluster in loaded.get("clusters", []):
        if not cluster.get("services"):
            errors.append(f"{cluster.get('cluster_id')} has no services")
        if len(cluster.get("time_range", [])) != 2:
            errors.append(f"{cluster.get('cluster_id')} has invalid time_range")

if errors:
    print("VALIDATION FAILED")
    for error in errors:
        print("-", error)
else:
    print("ALL ACCEPTANCE CRITERIA PASSED")
    print(f"clusters={loaded['output_clusters']}, reduction_ratio={loaded['reduction_ratio']}")

ALL ACCEPTANCE CRITERIA PASSED
clusters=3, reduction_ratio=0.85
